In [3]:
import pandas as pd
import numpy as np
import bnlp
import warnings
warnings.filterwarnings('ignore')

print(dir(bnlp))

['AsyncModelLoader', 'BNLPException', 'BasicTokenizer', 'BatchProcessor', 'BengaliCorpus', 'BengaliDoc2vec', 'BengaliDoc2vecTrainer', 'BengaliGlove', 'BengaliNER', 'BengaliPOS', 'BengaliWord2Vec', 'CRFTaggerTrainer', 'CleanText', 'LazyModelLoader', 'ModelLoadError', 'ModelNotFoundError', 'NLTKTokenizer', 'Pipeline', 'PipelineResult', 'PipelineStep', 'SentencepieceTokenizer', 'SentencepieceTrainer', 'Word2VecTraining', '__builtins__', '__cached__', '__doc__', '__file__', '__getattr__', '__loader__', '__name__', '__package__', '__path__', '__spec__', '__version__', 'clean_batch', 'cleantext', 'core', 'corpus', 'create_ner_pipeline', 'create_pos_pipeline', 'create_tokenization_pipeline', 'embed_batch', 'embedding', 'load_model_async', 'os', 'tag_batch', 'token_classification', 'tokenize_batch', 'tokenizer', 'utils']


In [4]:
from bnlp import BengaliPOS, BengaliNER, NLTKTokenizer

# Initialize
tokenizer = NLTKTokenizer()
pos_tagger = BengaliPOS()
ner_tagger = BengaliNER()

print("NLP tools loaded!")

NLP tools loaded!


In [5]:
# Load dataset
df = pd.read_csv('C:/Users/Riyad/projects/fake_news/step4_balanced.csv')
df['text'] = df['headline'].fillna('') + ' ' + df['content'].fillna('')
df = df.dropna(subset=['text', 'label'])

print(f" Data loaded: {len(df)}")
print(df['label'].value_counts())

 Data loaded: 15000
label
ai_fake      5000
fake         5000
authentic    5000
Name: count, dtype: int64


In [10]:
# Step by step debug
sample = df['text'].iloc[0]
print("Text:", sample[:100])

# Test word_tokenize
tokens = tokenizer.word_tokenize(sample)
print(f"Type: {type(tokens)}")
print(f"Tokens: {tokens}")

Text: মুশফিক-মোস্তাফিজকে ফিরিয়ে ব্যাটিংয়ে বাংলাদেশ শিরোনাম: ব্যাটিংয়ে ফিরিয়ে বাংলাদেশ মুশফিক-মোস্তাফিজকে  
Type: <class 'list'>
Tokens: ['মুশফিক-মোস্তাফিজকে', 'ফিরিয়ে', 'ব্যাটিংয়ে', 'বাংলাদেশ', 'শিরোনাম', ':', 'ব্যাটিংয়ে', 'ফিরিয়ে', 'বাংলাদেশ', 'মুশফিক-মোস্তাফিজকে', 'বিষয়বস্তু', ':', 'আফগানিস্তানের', 'বিপক্ষে', 'সুপার', 'ফোরের', 'এক', 'গুরুত্বহীন', 'ম্যাচে', 'বিশ্রামে', 'ছিলেন', 'মুশফিকুর', 'রহিম', 'ও', 'মোস্তাফিজুর', 'রহমান', '।', 'এখন', 'এই', 'দুজনকে', 'বাংলাদেশ', 'একাদশে', 'ফিরিয়েছে', '।', 'সুপার', 'ফোরের', 'আগেই', 'মুমিনুল', 'হক', 'ও', 'আবু', 'হায়দার', 'রনি', 'ব্যাটিংয়ে', 'বাংলাদেশকে', 'অভিন্ন', 'খেলায়', 'রেখে', 'বদল', 'করেছেন', '।', 'এশিয়া', 'কাপের', 'উ']


In [12]:
# Test POS tagger
tokens = tokenizer.word_tokenize(sample)
print(f"Token count: {len(tokens)}")

# Test POS
pos_tags = pos_tagger.tag(sample)
print(f"POS tags: {pos_tags[:5]}")

Token count: 54
POS tags: [('মুশফিক', 'JJ'), ('-', 'PU'), ('মোস্তাফিজকে', 'NC'), ('ফিরিয়ে', 'VM'), ('ব্যাটিংয়ে', 'NC')]


In [14]:
from tqdm import tqdm

# Extract NLP features
def extract_nlp_features(text):
    try:
        text = str(text)
        
        # Tokenize
        tokens = tokenizer.word_tokenize(text)
        
        # POS tagging (string দাও)
        pos_tags = pos_tagger.tag(text)
        
        # Count POS tags
        pos_counts = {}
        for word, tag in pos_tags:
            pos_counts[tag] = pos_counts.get(tag, 0) + 1
        
        # NER tagging (string দাও)
        ner_tags = ner_tagger.tag(text)
        ner_count = sum(1 for _, tag in ner_tags if tag != 'O')
        
        return {
            'token_count': len(tokens),
            'noun_count': pos_counts.get('NC', 0),
            'verb_count': pos_counts.get('VM', 0),
            'adjective_count': pos_counts.get('JJ', 0),
            'unique_pos_count': len(pos_counts),
            'ner_count': ner_count,
            'ner_ratio': ner_count / max(len(tokens), 1)
        }
    except:
        return {
            'token_count': 0,
            'noun_count': 0,
            'verb_count': 0,
            'adjective_count': 0,
            'unique_pos_count': 0,
            'ner_count': 0,
            'ner_ratio': 0.0
        }

# Test on 1 sample
result = extract_nlp_features(sample)
print(" Test result:")
print(result)

 Test result:
{'token_count': 54, 'noun_count': 23, 'verb_count': 6, 'adjective_count': 3, 'unique_pos_count': 10, 'ner_count': 20, 'ner_ratio': 0.37037037037037035}


In [15]:
# Apply to all data (this will take time)
print("Extracting NLP features...")


nlp_features = []

for i, row in tqdm(df.iterrows(), total=len(df), desc="Processing"):
    features = extract_nlp_features(row['text'])
    nlp_features.append(features)

# Convert to dataframe
nlp_df = pd.DataFrame(nlp_features)

# Merge with original
final_df = pd.concat([df.reset_index(drop=True), nlp_df], axis=1)

# Save
final_df.to_csv(
    'C:/Users/Riyad/projects/fake_news/step7_nlp_features.csv',
    index=False
)

print(f"\n NLP features saved!")
print(final_df[['label', 'token_count', 'noun_count', 'verb_count', 'ner_count']].groupby('label').mean())

Extracting NLP features...


Processing: 100%|██████████| 15000/15000 [01:28<00:00, 170.01it/s]



 NLP features saved!
           token_count  noun_count  verb_count  ner_count
label                                                    
ai_fake        62.8814     30.2496      5.6118     9.5756
authentic     323.8886    149.1514     32.8818    52.1372
fake          314.4464    144.2772     32.6788    42.5634
